# NFL Field-Goal Accuracy and Team Winning Percentage

"
       "**Research question:** Did NFL teams that made a higher percentage of their field goals also win more games during the 2025 regular season?

"
       "This notebook collects team kicking statistics and schedules, creates one analysis row per NFL team, documents the cleaning process, and evaluates the relationship with visualizations and correlation. This is an exploratory observational analysis; it cannot establish causation.

## 1. Setup and data collection

"
       "The primary collection method uses [`nflreadpy`](https://nflreadpy.nflverse.com/api/load_functions/): `load_team_stats` supplies regular-season team totals and `load_schedules` supplies final scores. The repository includes cached official nflverse CSV releases as a transparent fallback, so the work can still run if the package or network is unavailable.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from scipy.stats import linregress, pearsonr, spearmanr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
ASSETS = ROOT / "assets"
sns.set_theme(style="whitegrid", context="talk")

In [ ]:
try:
    import nflreadpy as nfl
    team_stats = nfl.load_team_stats(2025, summary_level="reg").to_pandas()
    schedules = nfl.load_schedules(2025).to_pandas()
    collection_method = "Loaded live with nflreadpy"
except (ImportError, OSError, ValueError):
    team_stats = pd.read_csv(RAW / "stats_team_reg_2025.csv")
    schedules = pd.read_csv(RAW / "games.csv", low_memory=False)
    collection_method = "Loaded cached official nflverse release files"

print(collection_method)
print("Team-stat source shape:", team_stats.shape)
print("Schedule source shape:", schedules.shape)

## 2. Cleaning and preparation

"
       "I filter to completed 2025 regular-season games, reshape each game into home-team and away-team observations, and calculate records from final scores. A tie contributes half a win to winning percentage. I then recalculate field-goal percentage from makes and attempts and perform a validated one-to-one join. I keep every team because all 32 belong to the population being described.

In [ ]:
regular = schedules.loc[
    (schedules["season"] == 2025)
    & (schedules["game_type"] == "REG")
    & schedules["home_score"].notna()
    & schedules["away_score"].notna()
].copy()

home = regular[["home_team", "home_score", "away_score"]].rename(
    columns={"home_team": "team", "home_score": "points_for", "away_score": "points_against"})
away = regular[["away_team", "away_score", "home_score"]].rename(
    columns={"away_team": "team", "away_score": "points_for", "home_score": "points_against"})
team_games = pd.concat([home, away], ignore_index=True)
team_games["win"] = (team_games.points_for > team_games.points_against).astype(int)
team_games["loss"] = (team_games.points_for < team_games.points_against).astype(int)
team_games["tie"] = (team_games.points_for == team_games.points_against).astype(int)

records = team_games.groupby("team", as_index=False).agg(
    wins=("win", "sum"), losses=("loss", "sum"), ties=("tie", "sum"), games_played=("team", "size"))
records["winning_percentage"] = (records.wins + 0.5 * records.ties) / records.games_played

In [ ]:
kicking = team_stats[["team", "fg_made", "fg_att", "fg_missed", "fg_blocked", "fg_long"]].copy()
kicking["field_goal_percentage"] = kicking.fg_made / kicking.fg_att

analysis = records.merge(kicking, on="team", how="inner", validate="one_to_one")
analysis["record"] = analysis.wins.astype(str) + "-" + analysis.losses.astype(str)
analysis.loc[analysis.ties > 0, "record"] += "-" + analysis.loc[analysis.ties > 0, "ties"].astype(str)
analysis = analysis.sort_values("field_goal_percentage", ascending=False)

assert analysis.shape[0] == 32
assert analysis.team.is_unique
assert analysis.isna().sum().sum() == 0
assert (analysis.games_played == 17).all()
print("Analysis rows:", len(analysis))
print("Missing values:", int(analysis.isna().sum().sum()))
analysis.head()

### Variables

"
       "- **Field-goal percentage (explanatory):** team field goals made divided by team field goals attempted during the regular season.
"
       "- **Winning percentage (outcome):** wins plus half of ties, divided by games played.
"
       "- **Supporting variables:** attempts, makes, misses, blocks, longest make, wins, losses, ties, and games played.

## 3. Analysis and visualization

"
       "Pearson correlation summarizes the strength of a linear relationship. I also calculate Spearman rank correlation as a robustness check because it is less dependent on a straight-line pattern.

In [ ]:
pearson_r, pearson_p = pearsonr(analysis.field_goal_percentage, analysis.winning_percentage)
spearman_rho, spearman_p = spearmanr(analysis.field_goal_percentage, analysis.winning_percentage)
reg = linregress(analysis.field_goal_percentage, analysis.winning_percentage)

print(f"Pearson r = {pearson_r:.3f}, p = {pearson_p:.3f}")
print(f"R-squared = {reg.rvalue**2:.3f}")
print(f"Spearman rho = {spearman_rho:.3f}, p = {spearman_p:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
sns.regplot(data=analysis, x="field_goal_percentage", y="winning_percentage", ci=None,
            scatter_kws={"s": 75, "alpha": .82, "color": "#007C91", "edgecolor": "white"},
            line_kws={"color": "#FCA311", "linewidth": 2.5}, ax=ax)
for row in analysis.itertuples():
    predicted = reg.intercept + reg.slope * row.field_goal_percentage
    if (row.field_goal_percentage in [analysis.field_goal_percentage.min(), analysis.field_goal_percentage.max()]
            or abs(row.winning_percentage - predicted) >= .19):
        ax.annotate(row.team, (row.field_goal_percentage, row.winning_percentage), xytext=(5, 5),
                    textcoords="offset points", fontsize=9)
ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set(title="Field-Goal Accuracy and Team Winning Percentage, 2025 NFL Regular Season",
       xlabel="Team field-goal percentage", ylabel="Team winning percentage")
plt.tight_layout(); plt.show()

The fitted line rises slightly, but teams are widely dispersed. Pearson **r = 0.14**, **p = 0.44**, and **R² = 0.02**. Field-goal percentage therefore explains only about 2% of the observed team-level variation in winning percentage, and this sample does not show a statistically significant linear relationship.

In [ ]:
ranked = analysis.sort_values("field_goal_percentage")
fig, ax = plt.subplots(figsize=(11, 12))
norm = plt.Normalize(analysis.winning_percentage.min(), analysis.winning_percentage.max())
cmap = plt.colormaps["viridis"]
bars = ax.barh(ranked.team, ranked.field_goal_percentage,
               color=cmap(norm(ranked.winning_percentage)), edgecolor="white")
ax.bar_label(bars, labels=[f"{x:.1%}" for x in ranked.field_goal_percentage], padding=3, fontsize=8)
ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.set_xlim(.67, 1.02)
ax.set(title="NFL Team Field-Goal Percentage in 2025", xlabel="Field-goal percentage", ylabel="Team")
colorbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, pad=.02)
colorbar.set_label("Winning percentage"); colorbar.ax.yaxis.set_major_formatter(PercentFormatter(1))
plt.tight_layout(); plt.show()

The Jets had the highest field-goal percentage (28 of 29, 96.6%) but finished 3-14. Denver finished 14-3 while making 87.5% of its attempts. These examples reinforce that accurate kicking is only one part of team success.

## 4. Conclusion

"
       "The 2025 data show a very weak positive relationship between field-goal percentage and winning percentage, but it is not statistically significant. I cannot conclude that better field-goal percentage caused teams to win more games—or that kicking accuracy has no value. Wins also reflect offense, defense, turnovers, injuries, schedule strength, coaching, and game context.

## 5. Limitations, ethics, and next steps

"
       "This analysis has only 32 observations and uses season aggregates. Raw percentage ignores kick distance, weather, stadium, pressure, blocks, attempt volume, score situation, and kicker changes. One season can also create temporal bias if its patterns are generalized to other eras. The public professional-performance data do not contain private personal information, but the results should not be used to make causal or individual personnel claims. A stronger next study would model attempt-level success using distance, environment, and game situation, then compare actual makes with expected makes.

## 6. References and AI disclosure

"
       "Berry, S. M., & Wood, C. (2004). The cold-foot effect. *Chance, 17*(4), 47-51. https://doi.org/10.1080/09332480.2004.10554926

"
       "Osborne, J. A., & Levine, R. A. (2017). Shrinkage estimation of NFL field goal success probabilities. *Journal of Sports Analytics, 3*(2), 129-146. https://doi.org/10.3233/JSA-16140

"
       "Pasteur, R. D., & Cunningham-Rhoads, K. (2014). An expectation-based metric for NFL field goal kickers. *Journal of Quantitative Analysis in Sports, 10*(1), 49-66. https://doi.org/10.1515/jqas-2013-0039

"
       "Data: nflverse, nflreadpy load functions, https://nflreadpy.nflverse.com/api/load_functions/

"
       "**AI disclosure:** OpenAI ChatGPT with Codex (GPT-5 family), accessed September 12, 2026, helped organize the project, draft and debug code, and revise writing. I reviewed the calculations, documentation, and conclusions and remain responsible for the submitted work. AI was not used as a data source.